# Chapter 1: Integers - Examples

Jupyter notebook accompanying the *Integers* chapter of Mathematical Preliminaries. 
Each cell demonstrates a concept from the text with concrete numeric examples.

## 1. Sieve of Eratosthenes

Generate all primes up to a given bound using the sieve described in the algorithm box.

In [ ]:
def sieve_of_eratosthenes(n: int) -> list[int]:
    """Generate a list of all prime numbers up to n."""
    if n < 2:
        return []
    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False  # 0 and 1 are not prime
    for p in range(2, int(n**0.5) + 1):
        if is_prime[p]:
            for i in range(p * p, n + 1, p):
                is_prime[i] = False
    return [num for num, prime in enumerate(is_prime) if prime]

print(sieve_of_eratosthenes(50))
print(f"There are {len(sieve_of_eratosthenes(1000))} primes below 1000")

## 2. Fundamental Theorem of Arithmetic

Factor a number into its unique prime power factorization:  
$n = p_1^{k_1} p_2^{k_2} \cdots p_\ell^{k_\ell}$

In [ ]:
def factorize(n: int) -> dict[int, int]:
    """Return dict of {prime: exponent} for the factorization of n."""
    factors = {}
    d = 2
    while d * d <= n:
        while n % d == 0:
            factors[d] = factors.get(d, 0) + 1
            n //= d
        d += 1 if d == 2 else 2
    if n > 1:
        factors[n] = factors.get(n, 0) + 1
    return factors

n = 360
f = factorize(n)
print(f"{n} = " + " ".join(f"{p}^{k}" for p, k in sorted(f.items())))

# Number of positive divisors: d(n) = prod(k_i + 1)
from functools import reduce
num_divisors = reduce(lambda x, y: x * y, [k + 1 for k in f.values()], 1)
print(f"Number of divisors of {n}: tau({n}) = {num_divisors}")

## 3. GCD and LCM via FTA

Using the prime factorizations:
$\gcd(a,b) = \prod p_i^{\min(j_i,k_i)}$ and $\text{lcm}(a,b) = \prod p_i^{\max(j_i,k_i)}$, and $\gcd(a,b)\cdot\text{lcm}(a,b) = a\cdot b$.

In [ ]:
import math
a, b = 84, 30
g = math.gcd(a, b)
l = a * b // g
print(f"gcd({a},{b}) = {g}, lcm({a},{b}) = {l}")
print(f"Check: gcd*lcm = {g*l} = a*b = {a*b}")

## 4. Bezout's identity (Extended Euclidean Algorithm)

$\gcd(a,b) = sa + tb$ for some $s,t \in \mathbb{Z}$.

In [ ]:
def extended_gcd(a, b):
    """Return (g, s, t) such that a*s + b*t = g = gcd(a,b)."""
    if b == 0:
        return (a, 1, 0)
    g, s1, t1 = extended_gcd(b, a % b)
    return (g, t1, s1 - (a // b) * t1)

a, b = 84, 30
g, s, t = extended_gcd(a, b)
print(f"gcd({a},{b}) = {g} = ({s})*{a} + ({t})*{b}")
assert g == s * a + t * b

## 5. Modular arithmetic / Units

Compute multiplication tables, find units, and check $a\mathbb{Z} \subseteq b\mathbb{Z}$ iff $b | a$.

In [ ]:
def units(n):
    """Elements of Z_n^\times: numbers coprime to n."""
    return [a for a in range(n) if math.gcd(a, n) == 1]

for n in [7, 9, 12]:
    print(f"Units of Z_{n}: {units(n)}  (count = {len(units(n))} = phi({n}))")

# Check: m is a unit mod n iff gcd(m, n) == 1
print("\nIs 2 a unit mod 12? gcd(2,12)=", math.gcd(2,12))
print("Is 5 a unit mod 12? gcd(5,12)=", math.gcd(5,12))

## 6. Chinese Remainder Theorem

Solve $x \equiv r \pmod{a}$ and $x \equiv s \pmod{b}$ with $\gcd(a,b)=1$.

In [ ]:
def crt(r, a, s, b):
    """Find N with 0 <= N < a*b, N % a == r, N % b == s."""
    g, inv_a, _ = extended_gcd(a, b)  # inv_a * a == 1 (mod b)
    assert g == 1, "moduli must be coprime!"
    N = r + a * ((s - r) * inv_a % b)
    return N % (a * b)

N = crt(r=2, a=3, s=3, b=5)
print(f"N = {N}, checks: {N} % 3 = {N%3}, {N} % 5 = {N%5}")

# CRT is used in RSA: speed decryption via mod p and mod q
# e.g. p=61, q=53, n=p*q
p, q = 61, 53
n = p * q
print(f"n = {p}*{q} = {n}")

## 7. Euler's totient function

Compute $\phi(n)$, use multiplicativity $\phi(mn)=\phi(m)\phi(n)$ when $\gcd(m,n)=1$, and verify $\phi(p^k)=p^k-p^{k-1}$.

In [ ]:
def phi(n):
    result = n
    p = 2
    while p * p <= n:
        if n % p == 0:
            while n % p == 0:
                n //= p
            result -= result // p
        p += 1
    if n > 1:
        result -= result // n
    return result

for n in [7, 12, 100, 2**5, 3**3]:
    print(f"phi({n}) = {phi(n)}")

# Multiplicativity example
print("\nphi(4)*phi(9) =", phi(4), "*", phi(9), "=", phi(4)*phi(9), "= phi(36)", phi(36))

# phi(p^k) = p^k - p^(k-1)
print("phi(2^5) = 2^5 - 2^4 =", 32 - 16, "(matches)")

## 8. Fermat's little theorem & Euler's theorem

Verify $a^{p-1} \equiv 1 \pmod p$ and $a^{\phi(n)} \equiv 1 \pmod n$ for $\gcd(a,n)=1$.

In [ ]:
# Fermat's little theorem
p = 15485863  # a prime
a = 2
print(f"2^{p-1} mod {p} =", pow(a, p-1, p), "(should be 1)")

# Euler's theorem
n = 21
a = 5
print(f"5^phi({n}) = 5^{phi(n)} mod {n} =", pow(a, phi(n), n), "(should be 1, gcd(5,21)=1)")

## 9. Legendre & Jacobi symbols

The Legendre symbol $(a|p)$ tells whether $a$ is a quadratic residue mod $p$. We compute it via Euler's criterion:  
$(a|p) \equiv a^{(p-1)/2} \pmod p$.

In [ ]:
def legendre(a, p):
    """Legendre symbol (a|p) for odd prime p."""
    if a % p == 0:
        return 0
    val = pow(a, (p-1)//2, p)
    return 1 if val == 1 else -1

p = 7
for a in range(1, 7):
    sym = legendre(a, p)
    meaning = "QR" if sym == 1 else "QNR"
    print(f"({a}|{p}) = {sym}  ({meaning})")

# Check which actually have square roots
print("\nSquares mod 7:", sorted({(x*x) % 7 for x in range(1,7)}))

## 10. Miller-Rabin compositeness test

Probabilistic primality testing. For a composite $n$, the probability that one random base fails to detect it is $\le 1/4$.

In [ ]:
import random

def is_probable_prime(n, k=10):
    """Miller-Rabin. Returns True if n is probably prime."""
    if n < 2: return False
    for small in [2,3,5,7,11,13,17,19,23,29]:
        if n % small == 0:
            return n == small
    d, r = n-1, 0
    while d % 2 == 0:
        d //= 2; r += 1
    for _ in range(k):
        a = random.randrange(2, n-1)
        x = pow(a, d, n)
        if x in (1, n-1):
            continue
        for _ in range(r-1):
            x = pow(x, 2, n)
            if x == n-1:
                break
        else:
            return False
    return True

for n in [561, 1105, 1729, 17, 97, 1009]:
    print(f"{n}: probably_prime = {is_probable_prime(n)}")

# 561, 1105, 1729 are Carmichael numbers (composite but Fermat-pseudoprime to all bases)